# Research 2 — the SECOND dataset

A short, read-only look at the data used in Research 2: what SECOND contains, how its labels are
decoded, how the frozen train/val/test split is checked, and a few example tiles. Nothing is trained,
downloaded or written; test labels are never used for a decision.

**SECOND** (Yang et al., https://captain-whu.github.io/SCD/): 4662 bi-temporal 512×512 RGB tile pairs with a
semantic label map for each date — official train 2968 pairs, official test 1694 pairs, no validation split.

One-time setup (already done on the research machine, not repeated here):
`satchange.semantic.second.bootstrap_second` installs the release into `data/SECOND/{train,test}`
(the release nests `SECOND_train_set.rar` / `SECOND_total_test.zip`; extract them there manually),
`python src/scripts/second_audit.py data/SECOND <report.json>` writes the duplicate/leakage audit, and
`prepare_second_splits` writes `data/SECOND/second_split_manifest.json` (an existing manifest is never replaced).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from satchange.data.second_binary import verify_second_sources
from satchange.semantic.second import (
    SECOND_CLASSES, SECOND_COLORS, decode_label, discover_second, read_second_split_manifest,
)

RUN_VERIFY_HASHES = False   # re-hash every file listed in the manifest (takes minutes)
N_EXAMPLES = 3              # example tiles shown in section 3

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
SECOND_ROOT = PROJECT_ROOT / "data" / "SECOND"
layout = discover_second(SECOND_ROOT)
print({split: len(rows) for split, rows in layout.items()}, "labelled pairs on disk")

## 1. Classes

Labels are RGB colour maps; `decode_label` turns them into class IDs (unknown colours raise an error).
**0 = no change (white)**, 1–6 = land-cover classes. A pixel is *changed* when its label is not 0; the
time-1 and time-2 change areas coincide, so one binary change mask follows from either label.

In [ ]:
COLOUR_OF = {class_id: rgb for rgb, class_id in SECOND_COLORS.items()}
for class_id, name in enumerate(("no change", *SECOND_CLASSES)):
    print(f"{class_id}  {name:30s} RGB {COLOUR_OF[class_id]}")

## 2. Frozen split

Validation is carved once from official-train **groups** (15 %, seed 2099; near-duplicate/adjacent tiles stay
in one group). The official test is used unchanged and was read once, after the protocol was frozen.
Reading the manifest also checks its content hash.

In [ ]:
manifest = read_second_split_manifest(SECOND_ROOT / "second_split_manifest.json")
splits = {name: set(ids) for name, ids in manifest["splits"].items()}
print("manifest sha256:", manifest["manifest_sha256"])
print("split sizes    :", {name: len(ids) for name, ids in splits.items()})
names = list(splits)
assert all(not splits[a] & splits[b] for i, a in enumerate(names) for b in names[i + 1:]), "splits overlap"
assert splits["test"] == {row["sample_id"] for row in layout["test"]}, "test differs from official test"
print("no overlap between splits; test == official test")
if RUN_VERIFY_HASHES:
    print("all files match their recorded hashes:", verify_second_sources(SECOND_ROOT))

## 3. Examples

Before, after, and the decoded time-1 / time-2 labels (white = no change).

In [ ]:
def colourize(label_path: str) -> np.ndarray:
    ids = decode_label(label_path)
    rgb = np.zeros((*ids.shape, 3), np.uint8)
    for class_id, colour in COLOUR_OF.items():
        rgb[ids == class_id] = colour
    return rgb

rows = layout["train"][:N_EXAMPLES]
figure, axes = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)), squeeze=False)
for axis_row, row in zip(axes, rows, strict=True):
    panels = [Image.open(row["before"]), Image.open(row["after"]),
              colourize(row["before_semantic"]), colourize(row["after_semantic"])]
    for axis, panel, title in zip(axis_row, panels, ("before", "after", "label T1", "label T2"), strict=True):
        axis.imshow(panel)
        axis.set_title(f"{row['sample_id']} {title}", fontsize=8)
        axis.axis("off")
plt.tight_layout()
plt.show()

## 4. Preprocessing

* Native 512×512, RGB scaled to [0, 1] float32, labels decoded to class IDs; nothing is baked into files.
* Training only: synchronized random flips/rot90 of both images and labels. Never on validation or test.
* ImageNet mean/std normalization is a per-model loader option for the pretrained encoders, not a default.